Installing Libraries:

In [ ]:
!pip install opencv-python
!pip install matplotlib
!pip install numpy
!pip install scikit-learn

Libraries:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

Finding Height and Width of an Image:

In [ ]:
import cv2
import os

# Example sample image path from session 1
sample_image_path = '/content/drive/MyDrive/Datasets/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/ROIs/001/L_Fore/01.bmp'

# Load the image in grayscale
img = cv2.imread(sample_image_path, cv2.IMREAD_GRAYSCALE)

# Check if image was loaded successfully
if img is None:
    print("Image could not be loaded. Check the path.")
else:
    # Print its shape
    print("Image shape:", img.shape)

    # Print height and width
    height, width = img.shape
    print("Height:", height)
    print("Width:", width)


Preprocessing for Training:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

# === CONFIGURATION ===
BASE_PATH = r"/content/drive/MyDrive/Datasets/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/ROIs"
IMAGE_SIZE = (128, 60)
FINGER_LIST = ['L_Fore', 'L_Middle', 'L_Ring', 'R_Fore', 'R_Middle', 'R_Ring']
NUM_IMAGES = 5  # Images per finger
FUSION_METHOD = 'vstack'  # Can also use 'hstack' or '2x3'

fused_images = []
fused_labels = []

# === STEP 1: LOAD & FUSE FINGER IMAGES ===
def load_and_fuse_finger_images(subject_path, subject_id):
    subject_samples = []
    labels = []

    for img_idx in range(1, NUM_IMAGES + 1):  # 1 to 9
        finger_images = []
        complete = True
        print(f"\n➡️ Subject {subject_id} - Image {img_idx:02d}")

        for finger in FINGER_LIST:
            img_path = os.path.join(subject_path, finger, f"{img_idx:02d}.bmp")
            print(f"  📥 Loading: {img_path}")
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                print(f"  ❌ Missing image: {img_path}")
                complete = False
                break

            img = cv2.resize(img, IMAGE_SIZE)
            img_eq = exposure.equalize_hist(img).astype(np.float64)
            img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)
            finger_images.append(img_norm)

        if complete and len(finger_images) == 6:
            print(f"  🔄 Fusing 6 fingers using {FUSION_METHOD}")
            if FUSION_METHOD == 'vstack':
                fused_img = np.vstack(finger_images)
            elif FUSION_METHOD == 'hstack':
                fused_img = np.hstack(finger_images)
            elif FUSION_METHOD == '2x3':
                top = np.hstack(finger_images[:3])
                bottom = np.hstack(finger_images[3:])
                fused_img = np.vstack([top, bottom])
            else:
                raise ValueError("Unsupported fusion method.")

            subject_samples.append(fused_img)
            label = f"{subject_id}_img{img_idx:02d}"
            labels.append(label)
            print(f"  ✅ Fused image shape: {fused_img.shape}")
        else:
            print(f"  ⚠️ Skipped image {img_idx:02d}")

    return subject_samples, labels

# === STEP 2: LOAD ALL SUBJECTS ===
subject_dirs = sorted(os.listdir(BASE_PATH))
for subj in tqdm(subject_dirs, desc="Loading and Fusing Images"):
    subject_path = os.path.join(BASE_PATH, subj)
    if not os.path.isdir(subject_path):
        continue

    fused, labels = load_and_fuse_finger_images(subject_path, subj)
    fused_images.extend(fused)
    fused_labels.extend(labels)

print(f"\n✅ Total Fused Samples: {len(fused_images)}")
print(f"✅ Sample Fused Image Shape: {fused_images[0].shape}")
fused_labels = np.array(fused_labels)

# === STEP 3: COMPUTE 2DPCA ===
def compute_2dpca(images_2d, num_components):
    print("\n⚙️ Computing 2DPCA projection matrix...")
    n = len(images_2d)
    h, w = images_2d[0].shape
    mean_img = sum(images_2d) / n
    G_t = np.zeros((w, w))

    for i, img in enumerate(images_2d):
        A = img - mean_img
        G_t += A.T @ A
        if i < 3:
            print(f"  ➕ Sample {i+1} contribution added")

    G_t /= n
    eig_vals, eig_vecs = np.linalg.eigh(G_t)
    idx = np.argsort(-eig_vals)  # Descending
    eig_vecs = eig_vecs[:, idx[:num_components]]
    print(f"✅ Projection matrix shape: {eig_vecs.shape}")
    return eig_vecs

num_components = 47  # Adjust depending on accuracy vs. size
W = compute_2dpca(fused_images, num_components)

# === STEP 4: PROJECT INTO 2DPCA SPACE ===
projected_features = []
for i, img in enumerate(fused_images):
    feat = img @ W  # Project along 2D columns
    projected_features.append(feat)
    if i < 3:
        print(f"🧮 Projected shape of sample {i+1}: {feat.shape}")

# === STEP 5: FLATTEN FOR CLASSIFIER (e.g., kNN, SVM) ===
flat_features = np.array([feat.flatten() for feat in projected_features])
print(f"\n✅ Flattened feature matrix shape: {flat_features.shape}")


Testing:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

# === CONFIGURATION ===
BASE_PATH = r"/content/drive/MyDrive/Datasets/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/ROIs"
IMAGE_SIZE = (128, 60)
FINGER_LIST = ['L_Fore', 'L_Middle', 'L_Ring', 'R_Fore', 'R_Middle', 'R_Ring']
TEST_INDICES = [6, 7, 8, 9, 10]  # You can also use [8, 9] for multiple test images

test_data = []
test_labels = []
test_paths = []

# === LOAD TEST DATA USING STRATEGY 1 (FUSED) ===
subject_dirs = sorted(os.listdir(BASE_PATH))
for subj in tqdm(subject_dirs, desc="Preparing test data"):
    subject_path = os.path.join(BASE_PATH, subj)
    if not os.path.isdir(subject_path):
        continue

    for img_idx in TEST_INDICES:
        finger_images = []
        paths_used = []

        for finger in FINGER_LIST:
            img_path = os.path.join(subject_path, finger, f"{img_idx:02d}.bmp")
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                print(f"⚠️ Missing: {img_path}")
                continue

            print(f"✅ Using: {img_path}")
            img = cv2.resize(img, IMAGE_SIZE)
            img_eq = exposure.equalize_hist(img).astype(np.float64)
            img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)

            finger_images.append(img_norm)
            paths_used.append(img_path)

        if len(finger_images) == 6:
            fused_img = np.vstack(finger_images)  # Match training fusion method
            test_data.append(fused_img)
            test_labels.append(f"{subj}_img{img_idx:02d}")
            test_paths.append(paths_used)
        else:
            print(f"⚠️ Incomplete fusion: Subject {subj}, Image {img_idx}")

# === CONVERT TO ARRAYS ===
test_data = np.array(test_data)
test_labels = np.array(test_labels)

# === STEP X: PROJECT TEST IMAGES USING 2DPCA ===
proj_test_features = [img @ W for img in test_data]
flat_test_features = np.array([f.flatten() for f in proj_test_features])

print(f"\n✅ Projected test features shape: {flat_test_features.shape}")


Benchmarking:

In [ ]:
correct_matches = 0
total_tests = len(flat_test_features)

print("\n🔍 Starting classification using Manhattan distance...")

for i in range(total_tests):
    test_vec = flat_test_features[i]
    true_label = test_labels[i]  # e.g., "005_img08"

    # 📏 Manhattan distance to all training vectors
    distances = np.sum(np.abs(flat_features - test_vec), axis=1)

    # 🏆 Find the closest match
    min_index = np.argmin(distances)
    predicted_label = fused_labels[min_index]  # e.g., "005_img03"

    print(f"\nTest sample {i+1}:")
    print(f"  🎯 Predicted → {predicted_label}")
    print(f"  ✅ Actual    → {true_label}")

    # Extract subject IDs only (before '_')
    pred_id = predicted_label.split('_')[0]
    true_id = true_label.split('_')[0]

    if pred_id == true_id:
        correct_matches += 1
        print("  🟢 Match (Subject ID correct)")
    else:
        print("  🔴 Mismatch")

# 📈 Compute final accuracy
accuracy = (correct_matches / total_tests) * 100
print(f"\n🏁 Final recognition accuracy: {accuracy:.2f}% ({correct_matches}/{total_tests})")
